# 04. Step 2: Category & Sentiment Classification

**Aspect-Category-Opinion-Sentiment (ACOS) Quadruple Extraction**

This notebook trains and evaluates **Step 2** of the ACOS framework:
- **Model Architecture (`CategorySentiClassification`):** Uses BERT with candidate aspect and opinion span representations to jointly predict aspect categories and sentiment polarities for each candidate pair.
- **Model Checkpointing:** Saves the best fine-tuned model checkpoint (`pytorch_model.bin`, `config.json`, `vocab.txt`) to `checkpoints/step2_best/` based on validation Micro-F1.
- **Evaluation on Pipeline Candidate Pairs:** Evaluates candidate pairs generated from Step 1 (`[domain]_test_pair_1st.tsv`) and produces complete quadruple predictions (`result.txt`), training curves, and metric CSVs.

## 1. Environment & Module Imports

In [ ]:
!pip install -q pytorch-crf transformers huggingface_hub seaborn scikit-learn matplotlib pandas boto3
import os
import sys
import random
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm, trange

# 1. Detect if repository is present; if running in fresh Colab session, auto-clone repository
if not os.path.exists("Extract-Classify-ACOS") and not os.path.exists("../Extract-Classify-ACOS"):
    if not os.path.exists("ACOS"):
        print("📥 Cloning ACOS repository from GitHub into Colab environment...")
        !git clone https://github.com/haisyamalawwab/ACOS.git

# 2. Robustly locate base project directory across Colab & Local
if os.path.exists("Extract-Classify-ACOS"):
    base_project_dir = os.path.abspath(".")
elif os.path.exists("../Extract-Classify-ACOS"):
    base_project_dir = os.path.abspath("..")
elif os.path.exists("ACOS/Extract-Classify-ACOS"):
    base_project_dir = os.path.abspath("ACOS")
elif os.path.exists("/content/ACOS/Extract-Classify-ACOS"):
    base_project_dir = "/content/ACOS"
elif os.path.exists("/content/Extract-Classify-ACOS"):
    base_project_dir = "/content"
else:
    base_project_dir = os.path.abspath(".")

extract_dir = os.path.join(base_project_dir, "Extract-Classify-ACOS")
notebooks_dir = os.path.join(base_project_dir, "notebooks")

for p in [base_project_dir, extract_dir, notebooks_dir]:
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler, TensorDataset
from modeling import CategorySentiClassification
from bert_utils.tokenization import BertTokenizer
from bert_utils.optimization import BertAdam
from run_classifier_dataset_utils import processors, output_modes, convert_examples_to_features2nd
from dataset_utils import read_pair_gold
from eval_metrics import pair_eval

# 3. Import colab_utils with fallback download
try:
    from colab_utils import (
        setup_timestamped_run_dir, download_bert_pretrained, analyze_and_plot_eda,
        plot_training_history, export_benchmark_tables_and_plots,
        display_quadruple_dataframe, df_to_markdown, export_step_table,
        MarkdownReport, SubtaskMetricCapture, plot_subtask_metrics,
        features_step1, features_step2, pair_examples_from_file,
        resolve_eval_pair_file, unpack_model_output,
    )
except ModuleNotFoundError:
    import urllib.request
    print("⚠️ Downloading colab_utils.py fallback directly from GitHub...")
    raw_url = "https://raw.githubusercontent.com/haisyamalawwab/ACOS/main/notebooks/colab_utils.py"
    urllib.request.urlretrieve(raw_url, "colab_utils.py")
    from colab_utils import (
        setup_timestamped_run_dir, download_bert_pretrained, analyze_and_plot_eda,
        plot_training_history, export_benchmark_tables_and_plots,
        display_quadruple_dataframe, df_to_markdown, export_step_table,
        MarkdownReport, SubtaskMetricCapture, plot_subtask_metrics,
        features_step1, features_step2, pair_examples_from_file,
        resolve_eval_pair_file, unpack_model_output,
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚡ Active PyTorch Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU Model: {torch.cuda.get_device_name(0)}")
print(f"📂 Base project directory: {base_project_dir}")
print(f"📁 Extract & Model directory: {extract_dir}")


## 2. Configuration & Hyperparameters

In [ ]:
DOMAIN = "rest16"              # 'rest16' or 'laptop'
TASK_NAME = "categorysenti"
MODEL_TYPE = "categorysenti"
DO_TRAIN = True                # Set to False to skip training and evaluate saved checkpoint
DO_EVAL = True
MAX_SEQ_LENGTH = 128
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
LEARNING_RATE = 5e-5
NUM_TRAIN_EPOCHS = 15          # Default is 30, 15 is great for fast Colab training
WARMUP_PROPORTION = 0.1
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Pretrained BERT Directory
bert_model_dir = os.path.join(base_project_dir, "bert_base_uncased")
download_bert_pretrained(target_dir=bert_model_dir)

data_dir = extract_dir
results_base = os.path.join(base_project_dir, "results")
session_dirs = setup_timestamped_run_dir(base_dir=results_base, domain=DOMAIN)
step2_checkpoint_dir = session_dirs["step2_checkpoint"]

print(f"📁 Step 2 Checkpoint will be saved to: {step2_checkpoint_dir}")
plots_dir = session_dirs["plots"]
csv_dir = session_dirs["csv"]
md_dir = session_dirs["md"]
logs_dir = session_dirs["logs"]

rep = MarkdownReport(
    f"04 - Step 2: Klasifikasi Category & Sentiment [{DOMAIN.upper()}]",
    md_dir,
    filename="04_step2_klasifikasi.md",
    meta={
        "domain": DOMAIN, "epochs": NUM_TRAIN_EPOCHS,
        "batch_train": TRAIN_BATCH_SIZE, "lr": LEARNING_RATE,
        "max_seq_length": MAX_SEQ_LENGTH, "device": str(device),
        "session_dir": session_dirs["root"],
    },
)

df_cfg = pd.DataFrame([
    {"Parameter": "domain", "Nilai": DOMAIN},
    {"Parameter": "num_train_epochs", "Nilai": NUM_TRAIN_EPOCHS},
    {"Parameter": "train_batch_size", "Nilai": TRAIN_BATCH_SIZE},
    {"Parameter": "eval_batch_size", "Nilai": EVAL_BATCH_SIZE},
    {"Parameter": "learning_rate", "Nilai": LEARNING_RATE},
    {"Parameter": "max_seq_length", "Nilai": MAX_SEQ_LENGTH},
    {"Parameter": "device", "Nilai": str(device)},
])
rep.section("1. Konfigurasi")
export_step_table(df_cfg, name="step2_00_konfigurasi", csv_dir=csv_dir, md_dir=md_dir,
                  title=f"Konfigurasi Step 2 ({DOMAIN.upper()})")
rep.table(df_cfg, caption="Hyperparameter yang dipakai")


## 3. Data Loading & Feature Processing

In [ ]:
tokenizer = BertTokenizer.from_pretrained(bert_model_dir, do_lower_case=True)
processor = processors[TASK_NAME]()
label_list = processor.get_labels(DOMAIN)
num_labels = len(label_list[0])

print(f"Jumlah kelas gabungan CATEGORY#SENTIMENT: {num_labels}")

# Tabel daftar kelas (bukan sekadar print panjang)
df_kelas = pd.DataFrame({
    "Index": range(len(label_list[0])),
    "Label": label_list[0],
})
df_kelas["Category"] = df_kelas["Label"].str.rsplit("#", n=1).str[0]
df_kelas["Sentiment"] = df_kelas["Label"].str.rsplit("#", n=1).str[1]
rep.section("2. Ruang label")
export_step_table(df_kelas, name="step2_01_daftar_kelas", csv_dir=csv_dir, md_dir=md_dir,
                  title=f"Daftar Kelas CATEGORY#SENTIMENT ({DOMAIN.upper()}) - {num_labels} kelas",
                  notes="Kelas dibentuk dari perkalian kategori x 3 sentimen (0=negative, 1=neutral, 2=positive).",
                  max_rows_md=30)
rep.table(df_kelas.head(20), caption=f"20 dari {num_labels} kelas")

# Pilih sumber evaluasi: pasangan prediksi step 1 kalau ada, jika tidak pasangan gold.
tokenized_dir = os.path.join(data_dir, "tokenized_data")
eval_pair_file, pakai_1st = resolve_eval_pair_file(tokenized_dir, DOMAIN, prefer_1st=True)

# CategorySentiProcessor tidak punya metode untuk memilih file bebas, jadi
# example dibangun langsung dari file terpilih memakai _read_tsv/_create_examples.
eval_examples = pair_examples_from_file(processor, eval_pair_file, set_type="test")

# features_step2 membungkus convert_examples_to_features2nd(examples, label_list,
# max_seq_length, tokenizer, output_mode).
eval_features = features_step2(eval_examples, label_list, MAX_SEQ_LENGTH, tokenizer,
                               output_modes[TASK_NAME])

all_input_ids = torch.tensor([f.aspect_input_ids for f in eval_features], dtype=torch.long)
all_input_mask = torch.tensor([f.aspect_input_mask for f in eval_features], dtype=torch.long)
all_segment_ids = torch.tensor([f.aspect_segment_ids for f in eval_features], dtype=torch.long)
all_candidate_aspect = torch.tensor([f.candidate_aspect for f in eval_features], dtype=torch.long)
all_candidate_opinion = torch.tensor([f.candidate_opinion for f in eval_features], dtype=torch.long)
all_label_id = torch.tensor([f.label_id for f in eval_features], dtype=torch.float)
all_tokens_len = torch.tensor([f.tokens_len for f in eval_features], dtype=torch.long)

eval_data = TensorDataset(all_tokens_len, all_input_ids, all_input_mask, all_segment_ids,
                          all_candidate_aspect, all_candidate_opinion, all_label_id)
eval_dataloader = DataLoader(eval_data, sampler=SequentialSampler(eval_data),
                            batch_size=EVAL_BATCH_SIZE)

# Gold selalu dari file pair gold, karena itu acuan penilaian.
class ArgsProxy:
    def __init__(self):
        self.bert_model = bert_model_dir
        self.do_lower_case = True

proxy_args = ArgsProxy()
test_pair_gold_file = os.path.join(tokenized_dir, f"{DOMAIN}_test_pair.tsv")
with open(test_pair_gold_file, "r", encoding="utf-8") as f:
    eval_gold = read_pair_gold(f.readlines(), proxy_args)

print(f"Kandidat evaluasi: {len(eval_examples)} | Gold pair: {len(eval_gold[0])}")

df_sumber = pd.DataFrame([{
    "File_Kandidat_Evaluasi": os.path.basename(eval_pair_file),
    "Sumber_Kandidat": "prediksi step 1" if pakai_1st else "gold (bukan pipeline penuh)",
    "Jumlah_Kandidat": len(eval_examples),
    "File_Gold": os.path.basename(test_pair_gold_file),
    "Jumlah_Gold_Pair": len(eval_gold[0]),
}])
rep.section("3. Sumber data evaluasi")
export_step_table(df_sumber, name="step2_02_sumber_evaluasi", csv_dir=csv_dir, md_dir=md_dir,
                  title=f"Sumber Data Evaluasi Step 2 ({DOMAIN.upper()})",
                  notes=("Skor pipeline penuh hanya valid bila kandidat berasal dari prediksi step 1. "
                         "Bila memakai gold pair, angka yang muncul mengukur step 2 secara terisolasi."))
rep.table(df_sumber, caption="Sumber evaluasi")

# Distribusi label pada data evaluasi
label_freq = all_label_id.sum(dim=0).numpy()
df_lbl = pd.DataFrame({"Label": label_list[0], "Frekuensi_Gold": label_freq.astype(int)})
df_lbl = df_lbl[df_lbl["Frekuensi_Gold"] > 0].sort_values("Frekuensi_Gold", ascending=False).reset_index(drop=True)
export_step_table(df_lbl, name="step2_03_distribusi_label_eval", csv_dir=csv_dir, md_dir=md_dir,
                  title=f"Label CATEGORY#SENTIMENT Aktif pada Data Evaluasi ({DOMAIN.upper()})",
                  notes=f"{len(df_lbl)} dari {num_labels} kelas benar-benar muncul pada data evaluasi.",
                  max_rows_md=25)
rep.table(df_lbl.head(15), caption="15 label tersering")


## 4. Model Initialization: `CategorySentiClassification`

In [ ]:
model = CategorySentiClassification.from_pretrained(bert_model_dir, num_labels=num_labels)
model.to(device)
print(f"✅ Initialized CategorySentiClassification model ({sum(p.numel() for p in model.parameters()):,} parameters).")

## 5. Training Loop with Model Checkpointing
Fine-tune the model on Ground Truth training pairs and persist the best checkpoint to `checkpoints/step2_best/`.

In [ ]:
if DO_TRAIN:
    train_examples = processor.get_train_examples(data_dir, DOMAIN)
    train_features = features_step2(train_examples, label_list, MAX_SEQ_LENGTH, tokenizer,
                                    output_modes[TASK_NAME])

    tr_input_ids = torch.tensor([f.aspect_input_ids for f in train_features], dtype=torch.long)
    tr_input_mask = torch.tensor([f.aspect_input_mask for f in train_features], dtype=torch.long)
    tr_segment_ids = torch.tensor([f.aspect_segment_ids for f in train_features], dtype=torch.long)
    tr_candidate_aspect = torch.tensor([f.candidate_aspect for f in train_features], dtype=torch.long)
    tr_candidate_opinion = torch.tensor([f.candidate_opinion for f in train_features], dtype=torch.long)
    tr_label_id = torch.tensor([f.label_id for f in train_features], dtype=torch.float)
    tr_tokens_len = torch.tensor([f.tokens_len for f in train_features], dtype=torch.long)

    train_data = TensorDataset(tr_tokens_len, tr_input_ids, tr_input_mask, tr_segment_ids,
                               tr_candidate_aspect, tr_candidate_opinion, tr_label_id)
    train_dataloader = DataLoader(train_data, sampler=RandomSampler(train_data),
                                 batch_size=TRAIN_BATCH_SIZE)

    num_train_optimization_steps = len(train_dataloader) * NUM_TRAIN_EPOCHS

    param_optimizer = list(model.named_parameters())
    no_decay = ["bias", "LayerNorm.bias", "LayerNorm.weight"]
    optimizer_grouped_parameters = [
        {"params": [p for n, p in param_optimizer if not any(nd in n for nd in no_decay)], "weight_decay": 0.01},
        {"params": [p for n, p in param_optimizer if any(nd in n for nd in no_decay)], "weight_decay": 0.0},
    ]
    optimizer = BertAdam(optimizer_grouped_parameters, lr=LEARNING_RATE,
                         warmup=WARMUP_PROPORTION, t_total=num_train_optimization_steps)

    print(f"Mulai training step 2: {NUM_TRAIN_EPOCHS} epoch, {len(train_dataloader)} step/epoch")

    class ArgsHelper:
        def __init__(self):
            self.output_dir = logs_dir
            self.max_seq_length = MAX_SEQ_LENGTH

    eval_args = ArgsHelper()

    import logging
    logger = logging.getLogger("Step2")

    best_val_f1 = 0.0
    training_history = []
    step_loss_log = []

    for epoch in range(1, NUM_TRAIN_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        for step, batch in enumerate(tqdm(train_dataloader, desc=f"Epoch {epoch}/{NUM_TRAIN_EPOCHS}")):
            batch = tuple(t.to(device) for t in batch)
            _len, _ids, _mask, _seg_ids, _cand_a, _cand_o, _lbls = batch

            # CategorySentiClassification mengembalikan ([loss], [logits]).
            out = model(
                tokenizer, epoch,
                aspect_input_ids=_ids,
                aspect_token_type_ids=_seg_ids,
                aspect_attention_mask=_mask,
                candidate_aspect=_cand_a,
                candidate_opinion=_cand_o,
                label_id=_lbls,
            )
            loss, _ = unpack_model_output(out)

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total_loss += loss.item()
            step_loss_log.append({"epoch": epoch, "step": step + 1, "loss": loss.item()})

        avg_loss = total_loss / len(train_dataloader)

        model.eval()
        val_res = pair_eval(epoch, eval_args, logger, tokenizer, model, eval_dataloader,
                            eval_gold, label_list, device, TASK_NAME, eval_type="test")

        val_p = val_res.get("precision", 0.0)
        val_r = val_res.get("recall", 0.0)
        val_f1 = val_res.get("micro-F1", 0.0)

        print(f"Epoch {epoch:02d} | loss {avg_loss:.4f} | P {val_p*100:.2f}% | "
              f"R {val_r*100:.2f}% | micro-F1 {val_f1*100:.2f}%")

        training_history.append({
            "epoch": epoch, "loss": avg_loss,
            "precision": val_p, "recall": val_r, "micro-F1": val_f1,
        })

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            print(f"  -> micro-F1 terbaik baru ({best_val_f1*100:.2f}%), menyimpan checkpoint")
            torch.save(model.state_dict(), os.path.join(step2_checkpoint_dir, "pytorch_model.bin"))
            model.config.to_json_file(os.path.join(step2_checkpoint_dir, "config.json"))
            tokenizer.save_vocabulary(step2_checkpoint_dir)
            with open(os.path.join(step2_checkpoint_dir, "checkpoint_metadata.json"), "w") as mf:
                json.dump({
                    "epoch": epoch, "best_micro_f1": best_val_f1,
                    "precision": val_p, "recall": val_r,
                    "domain": DOMAIN, "task": "Step2_Category_Sentiment_Classification",
                }, mf, indent=2)

    plot_history_path = os.path.join(plots_dir, "04_step2_training_loss_f1_curve.png")
    csv_history_path = os.path.join(csv_dir, "step2_training_history.csv")
    plot_training_history(training_history, task_name="Step 2 (Category-Sentiment)",
                          output_plot_path=plot_history_path, output_csv_path=csv_history_path)

    df_hist = pd.DataFrame(training_history)
    df_hist_pct = df_hist.copy()
    for c in ["precision", "recall", "micro-F1"]:
        df_hist_pct[c] = (df_hist_pct[c] * 100).round(2)

    rep.section("4. Riwayat training per epoch")
    export_step_table(df_hist_pct, name="step2_04_riwayat_epoch", csv_dir=csv_dir, md_dir=md_dir,
                      title=f"Riwayat Training Step 2 per Epoch ({DOMAIN.upper()})",
                      notes="Metrik dihitung pada level quadruple lengkap lewat pair_eval.",
                      max_rows_md=NUM_TRAIN_EPOCHS)
    rep.table(df_hist_pct, max_rows=NUM_TRAIN_EPOCHS, caption="Metrik per epoch")

    df_steps = pd.DataFrame(step_loss_log)
    export_step_table(df_steps.groupby("epoch")["loss"].describe().reset_index(),
                      name="step2_05_statistik_loss_per_epoch", csv_dir=csv_dir, md_dir=md_dir,
                      title=f"Statistik Loss per Epoch ({DOMAIN.upper()})",
                      max_rows_md=NUM_TRAIN_EPOCHS)
    df_steps.to_csv(os.path.join(csv_dir, "step2_loss_per_step.csv"), index=False, encoding="utf-8")

    best_row = df_hist_pct.loc[df_hist_pct["micro-F1"].idxmax()]
    rep.section("5. Epoch terbaik").kv({
        "epoch": int(best_row["epoch"]),
        "loss": f"{best_row['loss']:.4f}",
        "precision": f"{best_row['precision']:.2f}%",
        "recall": f"{best_row['recall']:.2f}%",
        "micro-F1": f"{best_row['micro-F1']:.2f}%",
        "checkpoint": step2_checkpoint_dir,
    })
    print(f"Epoch terbaik: {int(best_row['epoch'])} (micro-F1 {best_row['micro-F1']:.2f}%)")
else:
    print("DO_TRAIN=False, training dilewati. Lanjut ke pemuatan checkpoint.")
    training_history = []


## 6. Standalone Checkpoint Loading & Full Quadruple Evaluation
Load the saved checkpoint from `checkpoints/step2_best/` and compute final metrics across all 15 subtasks.

In [ ]:
print(f"Memuat checkpoint step 2 terbaik dari: {step2_checkpoint_dir}")
model = CategorySentiClassification.from_pretrained(step2_checkpoint_dir, num_labels=num_labels)
model.to(device)
model.eval()

class ArgsHelper:
    def __init__(self):
        self.output_dir = logs_dir
        self.max_seq_length = MAX_SEQ_LENGTH

eval_args = ArgsHelper()

import logging
logger = logging.getLogger("Step2_Final")

# pair_eval menghitung metrik 15 sub-task tetapi hanya menulisnya ke logger.
# SubtaskMetricCapture menangkap baris log itu supaya angkanya bisa ditabelkan.
with SubtaskMetricCapture(logger) as cap:
    final_res = pair_eval("best_checkpoint", eval_args, logger, tokenizer, model, eval_dataloader,
                          eval_gold, label_list, device, TASK_NAME, eval_type="test")

df_final = pd.DataFrame([{
    "Metrik": k, "Nilai": v, "Persen": round(v * 100, 2) if isinstance(v, float) else v,
} for k, v in final_res.items()])

rep.section("6. Hasil akhir quadruple pada test set")
export_step_table(df_final, name="step2_06_hasil_quadruple_final", csv_dir=csv_dir, md_dir=md_dir,
                  title=f"Hasil Akhir Ekstraksi Quadruple ({DOMAIN.upper()})",
                  notes=("Skor pipeline penuh hanya bila kandidat berasal dari prediksi step 1 "
                         f"(sumber saat ini: {'prediksi step 1' if pakai_1st else 'gold pair'})."))
rep.table(df_final, caption="Metrik quadruple")

print("\nHasil akhir quadruple pada test set:")
for k, v in final_res.items():
    print(f"  {k}: {v*100:.2f}%")

# Metrik per sub-task hasil tangkapan log
df_sub = cap.to_frame()
if df_sub.empty:
    print("\n[catatan] Metrik sub-task tidak tertangkap dari log pair_eval.")
    rep.section("7. Metrik per sub-task").text(
        "Metrik sub-task tidak tertangkap. Pastikan level logger memungkinkan pesan INFO."
    )
else:
    df_sub_pct = df_sub.copy()
    for c in ["Precision", "Recall", "Micro_F1"]:
        df_sub_pct[c] = (df_sub_pct[c] * 100).round(2)

    rep.section("7. Metrik per sub-task (15 kombinasi elemen)")
    export_step_table(df_sub_pct, name="step2_07_metrik_15_subtask", csv_dir=csv_dir, md_dir=md_dir,
                      title=f"Metrik 15 Sub-Task Hasil Evaluasi Nyata ({DOMAIN.upper()})",
                      notes=("Diambil langsung dari keluaran pair_eval, bukan angka yang ditulis manual. "
                             "N_Elements = jumlah elemen quadruple yang dievaluasi bersamaan."),
                      max_rows_md=20)
    rep.table(df_sub_pct, max_rows=20, caption="Metrik per sub-task")

    p_sub = os.path.join(plots_dir, "04b_step2_subtask_f1.png")
    plot_subtask_metrics(df_sub, p_sub,
                         title=f"[{DOMAIN.upper()}] Micro-F1 per Sub-Task (evaluasi nyata)")
    rep.image(p_sub, "Micro-F1 per sub-task")

    # Simpan juga sebagai JSON agar notebook 05 bisa memakai angka nyata
    subtask_json = os.path.join(logs_dir, "subtask_metrics.json")
    with open(subtask_json, "w", encoding="utf-8") as jf:
        json.dump(cap.to_dict(), jf, indent=2)
    print(f"[metrik] Metrik sub-task disimpan: {subtask_json}")

result_file = os.path.join(logs_dir, "result.txt")
if os.path.exists(result_file):
    print(f"[file] Prediksi lengkap: {result_file}")
    rep.text(f"File prediksi lengkap: `{result_file}`")


## 7. Display Step 2 Training Loss & Metrics Curve

In [ ]:
from IPython.display import Image, display

step2_plots = [
    ("04_step2_training_loss_f1_curve.png", "Kurva loss training & metrik quadruple"),
    ("04b_step2_subtask_f1.png", "Micro-F1 per sub-task"),
]

rep.section("8. Visualisasi step 2")
for fname, caption in step2_plots:
    path = os.path.join(plots_dir, fname)
    if os.path.exists(path):
        print(f"[plot] {caption}")
        display(Image(path))
    else:
        print(f"[plot] Tidak ditemukan (dilewati): {fname}")

rep.text(f"Sesi: `{session_dirs['root']}`")
report_path = rep.save()

print(f"\nLaporan Markdown step 2: {report_path}")
print("Lanjut ke '05_ACOS_Evaluation_and_Interactive_Inference.ipynb'.")
